## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import MDAnalysis as mda
import os
import MDAnalysis as mda
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns

## Finding Optimal Clusters using Elbow Method

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42) 
    kmeans.fit(reduced_df)
    inertia.append(kmeans.inertia_) 
    
# Plotting the Elbow Curve
plt.figure(figsize=(8, 6))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.show()

## Clustering

In [ ]:
optimal_k = x #(insert value froom elbow method) 

# Apply KMeans with the optimal k
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
kmeans.fit(reduced_df)
cluster_labels = kmeans.labels_

# Adding cluster labels
reduced_df['cluster'] = cluster_labels

# Visualize the clustering
plt.figure(figsize=(8, 6))
for cluster in range(optimal_k):
    cluster_data = reduced_df[reduced_df['cluster'] == cluster]
    plt.scatter(cluster_data.iloc[:, 0], cluster_data.iloc[:, 1], label=f"Cluster {cluster}")

plt.title("KMeans Clustering Results")
plt.xlabel("latent 1")
plt.ylabel("latent 2")
plt.legend(title="Clusters", loc="upper right")
plt.show()

## Representative frame 

In [ ]:
centroids = kmeans.cluster_centers_ 

representative_frames = []

for cluster_id in range(optimal_k):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    if len(cluster_indices) == 0:
        print(f"Cluster {cluster_id} is empty.")
        continue
    cluster_points = reduced_df.iloc[cluster_indices].values[:, :2]
    distances = np.linalg.norm(cluster_points - centroids[cluster_id], axis=1)
    closest_index = cluster_indices[np.argmin(distances)]
    representative_frames.append(closest_index)

# Print the indices of representative frames
representative_data_df = combined_df.iloc[representative_frames]
print("Representative Frames Data:")
representative_data_df

## Cluster average

In [ ]:
combined_df['cluster'] = reduced_df['cluster']
cluster_average = combined_df.groupby('cluster').mean().reset_index()

# Print the averages
cluster_average.drop(['Frame','Trajectory'], axis =1)

## Representative structures

In [ ]:
# Create directory
output_dir = "rep_pdb_encoder"
os.makedirs(output_dir, exist_ok=True)

topology = "topology.pdb" 

# Function to save PDBs from representative frames
def save_representative_frames(representative_list, combined_df):
    for frame_index in representative_list:
        traj_id = combined_df.loc[frame_index, 'Trajectory']
        frame_number = combined_df.loc[frame_index, 'Frame']
        trajectory_file = f"{traj_id}.xtc" 
        u = mda.Universe(topology, trajectory_file)
        u.trajectory[frame_number]  
        
        # Construct the output filename for the PDB file
        output_filename = os.path.join(output_dir, f"frame_{traj_id}_{frame_number}.pdb")
        
        # Write the current frame to a PDB file
        with mda.Writer(output_filename, multiframe=False) as w:
            w.write(u.atoms)

        print(f"Saved frame {frame_number} from trajectory {traj_id} to {output_filename}")

# Save representative frames
save_representative_frames(representative_frames, combined_df)

## Classification

In [ ]:
X = reduced_df.drop('cluster', axis=1)  # Exclude the cluster labels
y = reduced_df['cluster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define classifiers
classifiers = {
    'SVM': SVC(),
    'KNN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier()
}

accuracy_results = {}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_results[name] = accuracy
    print(f'{name} Accuracy: {accuracy}')
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

## Cross validation

In [ ]:
from sklearn.model_selection import cross_val_score

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X, y, cv=5)  # 5-fold cross-validation
    print(f'{name} Cross-Validation Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

## learning_curve

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

def plot_learning_curve(estimator, X, y, title):
    train_sizes, train_scores, test_scores = learning_curve(estimator, X, y, cv=5, scoring='accuracy')
    train_scores_mean = np.mean(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)

    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_scores_mean, label="Training Score")
    plt.plot(train_sizes, test_scores_mean, label="Cross-Validation Score")
    plt.title(title)
    plt.xlabel("Training Set Size")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

for name, clf in classifiers.items():
    plot_learning_curve(clf, X, y, title=f'{name} Learning Curve')